In [0]:
###

In [0]:


spark.conf.set(
    "fs.azure.account.auth.type.nyctaxistorages.dfs.core.windows.net",
    "OAuth"
)

spark.conf.set(
    "fs.azure.account.oauth.provider.type.nyctaxistorages.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    "fs.azure.account.oauth2.client.id.nyctaxistorages.dfs.core.windows.net",
    "<YOUR_CLIENT_ID>"
)

spark.conf.set(
    "fs.azure.account.oauth2.client.secret.nyctaxistorages.dfs.core.windows.net",
    "<YOUR_CLIENT_SECRET>"
)

spark.conf.set(
    "fs.azure.account.oauth2.client.endpoint.nyctaxistorages.dfs.core.windows.net",
    "https://login.microsoftonline.com/<YOUR_TENANT_ID>/oauth2/token"
)

In [0]:
account_fqdn = "nyctaxistorages.dfs.core.windows.net"
placeholder_fqdn = "<storage-account>.dfs.core.windows.net"

spark.conf.set(f"fs.azure.account.oauth.provider.type.{account_fqdn}", spark.conf.get(f"fs.azure.account.oauth.provider.type.{placeholder_fqdn}"))
spark.conf.set(f"fs.azure.account.oauth2.client.id.{account_fqdn}", spark.conf.get(f"fs.azure.account.oauth2.client.id.{placeholder_fqdn}"))
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{account_fqdn}", spark.conf.get(f"fs.azure.account.oauth2.client.secret.{placeholder_fqdn}"))
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{account_fqdn}", spark.conf.get(f"fs.azure.account.oauth2.client.endpoint.{placeholder_fqdn}"))

dbutils.fs.ls('abfss://bronze@nyctaxistorages.dfs.core.windows.net/')

#Data Reading

###imorting libraries

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

###Reading CSV Data

###Trip Type Data

In [0]:
df_trip_type = spark.read.format('csv')\
                    .option("header", "true")\
                    .option("inferSchema", "true")\
                    .load('abfss://bronze@nyctaxistorages.dfs.core.windows.net/trip_type')

In [0]:
df_trip_type.display()

In [0]:
df_trip_zone = spark.read.format('csv')\
                    .option("header", "true")\
                    .option("inferSchema", "true")\
                    .load('abfss://bronze@nyctaxistorages.dfs.core.windows.net/trip_zone')

In [0]:
df_trip_zone.display()

###Trip Data

In [0]:
df_trip = spark.read.format('parquet')\
                    .option("header", "true")\
                    .schema(my_schema)\
                    .option("recursiveFileLookup", True)\
                    .load('abfss://bronze@nyctaxistorages.dfs.core.windows.net/trips2023data')

In [0]:

my_schema = '''
                VendorID BIGINT,
lpep_pickup_datetime TIMESTAMP,
lpep_dropoff_datetime TIMESTAMP,
store_and_fwd_flag STRING,
RatecodeID BIGINT,
PULocationID BIGINT,
DOLocationID BIGINT,
passenger_count BIGINT,
trip_distance DOUBLE,
fare_amount DOUBLE,
extra DOUBLE,
mta_tax DOUBLE,
tip_amount DOUBLE,
tolls_amount DOUBLE,
ehail_fee DOUBLE,
improvement_surcharge DOUBLE,
total_amount DOUBLE,
payment_type BIGINT,
trip_type BIGINT,
congestion_surcharge DOUBLE
'''

In [0]:
df_trip.display()

#Data Transformation

#taxi_trip_type

In [0]:
df_trip_type.display()

In [0]:
df_trip_type = df_trip_type.withColumnRenamed('description','type_description')

In [0]:
df_trip_type.write.format('parquet')\
                    .mode('append')\
                    .option("path",'abfss://silver@nyctaxistorages.dfs.core.windows.net/trip_type')\
                    .save()

###Trip Zone

In [0]:
df_trip_zone.display()

In [0]:
df_trip_zone = df_trip_zone.withColumn('zone1', split('Zone','/')[0])

In [0]:
df_trip_zone = df_trip_zone.withColumn('zone2', split('Zone','/')[1])

In [0]:
df_trip_zone.display()

In [0]:
df_trip_zone.write.format('parquet')\
                    .mode('append')\
                    .option("path",'abfss://silver@nyctaxistorages.dfs.core.windows.net/trip_zone')\
                    .save()



In [0]:
df_trip.display()

In [0]:
df_trip = df_trip.withColumn('trip_date', to_date('lpep_pickup_datetime'))\
                .withColumn('trip_year',year('lpep_pickup_datetime'))\
                .withColumn('trip_month',month('lpep_pickup_datetime'))


In [0]:
df_trip.display()

In [0]:
df_trip = df_trip.select('VendorID','PULocationID','DOLocationID','fare_amount','total_amount')

In [0]:
df_trip.display()

In [0]:
df_trip.write.format('parquet')\
                    .mode('append')\
                    .option("path",'abfss://silver@nyctaxistorages.dfs.core.windows.net/trip2023data')\
                    .save()

#Analysis

In [0]:
display(df_trip)

Databricks visualization. Run in Databricks to view.